In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Every Axis Means Something

In potential-energy.ipynb we integrated a whole bunch of pendulums at once and got back an array `y` with three axes: time, state (angle or angular velocity), and which pendulum.

When you're used to matrices it's tempting to ask: which two of those axes represent a matrix and what does the extra one mean?

The answer is to free your mind by unasking the question.

Each axis is a list of the possible answers to one question.  *Which time?  Which part of the state?  Which pendulum?*  To get a single number out of `y` you have to answer all three, just like you need three coordinates to pick out a point in 3D space.

If you only answer *some* of the questions, `numpy` hands you back everything that's consistent with your answers, and the questions you didn't answer become the axes of the result.  Ask "what was every pendulum's angle at every time?" and you get a (time, pendulum) array.  Ask "what did pendulum 3 do?" and you get a (time, state) array.  They are both rectangular grids of numbers.  Neither one is more deserving of the title of "The Matrix™" than the other.  They're answers to different questions!

That's what this whole assignment is about: turning questions into indices, and predicting the shape of what comes back.

# The Rules

Inside the square brackets we write one entry per axis, starting from the left.  Each entry either pins its axis down or leaves it open.

| entry | means | what happens to that axis |
|---|---|---|
| `3` | just number 3 | gone |
| `-1` | just the last one | gone |
| `:` | all of them | kept |
| `2:5` | numbers 2, 3, and 4 (*not* 5!) | kept, but shorter |
| `::10` | every 10th one, starting from 0 | kept, but shorter |
| `...` | all of them, for every axis I didn't mention | all kept |

Any axes you leave off the right-hand end count as `:`, so `y[3]` means the same thing as `y[3, :, :]`.

And from the broadcasting assignment, `np.newaxis` is one more thing you can put inside the brackets: it *adds* a brand-new axis of length 1.

Every exercise asks you for two things: the slice, and your **prediction** of its shape.  Make the prediction *before* you run the cell!  This little function compares them.

In [ ]:
def check_shape(answer, prediction):
    '''
    Compares the shape of answer to the shape you predicted.
    '''
    if np.shape(answer) == prediction:
        print(f'✓ {prediction}')
    else:
        print(f'✗ you predicted {prediction} but the shape is {np.shape(answer)}')

Write your predictions in terms of lengths, like `(len(t), len(E))`, rather than bare numbers.  That way your prediction says what each axis *means*, not just how long it is.

# From Numbers to Arrays

A single number has no axes at all.  There's nothing left to ask, so its shape is the empty tuple `()`.

In [ ]:
reading = 28.3   # degrees Celsius
print(np.shape(reading))

Now suppose a weather station records the temperature once an hour for a day.  That's a vector: one axis, and that axis *means* the hour of the day.

In [ ]:
rng = np.random.default_rng(320)

hours = 24
hour = np.arange(hours)

# Warmest in the middle of the afternoon, plus a little noise.
today = 28 + 2 * np.sin(2*np.pi * (hour - 9) / hours) + rng.normal(0, 0.3, size=hours)
print(today.shape)

One index picks one hour and leaves no axes behind.  A range picks *some* hours and keeps the hour axis, just shorter.

In [ ]:
noon = ...                           # the temperature at noon (hour 12)
check_shape(noon, ...)               # replace ... with your prediction, as a tuple

afternoon = ...                      # the temperatures from noon up to (but not including) 6pm
check_shape(afternoon, ...)

every_third_hour = ...               # midnight, 3am, 6am, ...
check_shape(every_third_hour, ...)

last_reading = ...                   # the last reading of the day, using a negative index
check_shape(last_reading, ...)

Now a whole week.  Every day has the same hours, so a week of readings needs two answers to get a temperature: *which day?* and *which hour?*  That's a matrix, with axes (day, hour).

The rows aren't special and the columns aren't special.  We could just as well have stored it as (hour, day).  All that matters is that we remember which axis means what.

In [ ]:
days = 7

week = 28 + 2 * np.sin(2*np.pi * (hour - 9) / hours) + rng.normal(0, 0.3, size=(days, hours))
print(week.shape)

Say the week starts on Monday, so day 0 is Monday.

In [ ]:
wednesday = ...                      # every reading from Wednesday
check_shape(wednesday, ...)

noon_every_day = ...                 # the noon reading, every day
check_shape(noon_every_day, ...)

weekend_afternoons = ...             # Saturday and Sunday, noon up to 6pm
check_shape(weekend_afternoons, ...)

When you hand `plt.plot` a 2D array, it lines the array's **first** axis up with the $x$ values and draws one curve for each entry along the **second** axis.

Plot the temperature as a function of the hour, with one curve for each day.  `week` has its axes in the order (day, hour), which is the wrong way around for `plt.plot`.  `week.T` (the *transpose*) is the same numbers with the order of the axes reversed, (hour, day).

In [ ]:
fig, ax = plt.subplots()

... # one curve per day, temperature vs. hour

ax.set_xlabel('hour')
ax.set_ylabel('temperature (°C)')

Now three weather stations, one on each island.  Three axes: (station, day, hour).

This isn't "a stack of three matrices."  It's the temperature as a function of three things.  To get one number you need to say which station, which day, and which hour.

In [ ]:
stations = ['St. Croix', 'St. Thomas', 'St. John']

temperature = 28 + 2 * np.sin(2*np.pi * (hour - 9) / hours) + rng.normal(0, 0.3, size=(len(stations), days, hours))
print(temperature.shape)

In [ ]:
st_thomas_wednesday_noon = ...       # St. Thomas, Wednesday, noon
check_shape(st_thomas_wednesday_noon, ...)

everywhere_at_noon = ...             # every station, every day, at noon
check_shape(everywhere_at_noon, ...)

st_john = ...                        # every reading from St. John
check_shape(st_john, ...)

sunday_3pm = ...                     # every station at 3pm on Sunday
check_shape(sunday_3pm, ...)

**Two of your answers are matrices.  For each one, say what its two axes mean.  Which one is "the matrix" inside `temperature`?**

Nothing stops us here.  If the stations recorded for a whole year we could add a week axis and have (week, station, day, hour).  Measure the humidity too and we could add another axis for (temperature, humidity).  An array has as many axes as there are questions you need to answer to pin down one number.

# Setup

Now let's go back to the pendulums.  Here's the code from potential-energy.ipynb and symplectic-integrators.ipynb.

In [ ]:
class FirstOrderODE:
    '''
    A first-order ODE specifies the rate of change of a state y(t)
    in terms of
     - the state itself, y
     - the time, t.

    The generic form for an ODE is

    dy
    --  = f(t, y)
    dt

    '''

    def f(self, t, y):
        raise NotImplementedError('The generic ODE has an unspecified slope function f.')

    def exact_solution(self, t, *args):
        raise NotImplementedError('The exact solution is not known (or at least not provided).')

In [ ]:
class SecondOrderODE(FirstOrderODE):
    '''
    A second-order ODE specifies the acceleration

    d^2x
    ----  = a(t, x, v)
    dt^2

    where v = dx/dt is the velocity.  Treating the state as y = (x, v) turns it into
    a first-order ODE,

    d  ( x )   (    v     )
    -- (   ) = (          )
    dt ( v )   ( a(t,x,v) )

    so a subclass only needs to provide the acceleration.
    '''

    def acceleration(self, t, x, v):
        raise NotImplementedError('The generic second-order ODE has an unspecified acceleration.')

    def f(self, t, y):
        # Unpack the state into its two components,
        x, v = y
        # and pack their rates of change back up the same way.
        return np.array([v, self.acceleration(t, x, v)])

In [ ]:
class ConservativeSystem(SecondOrderODE):
    '''
    A conservative system has a potential energy V(x), and its acceleration is

    a = -V'(x)

    which doesn't depend on the velocity or the time.  Its energy

    E = 1/2 v^2 + V(x)

    is conserved.  A subclass provides the potential and the acceleration (minus its slope!).
    Everything is per unit mass.
    '''

    def potential(self, x):
        raise NotImplementedError('The generic conservative system has an unspecified potential.')

    def energy(self, x, v):
        return 0.5 * v**2 + self.potential(x)

In [ ]:
class Pendulum(ConservativeSystem):
    '''
    A simple pendulum has potential V = ω^2 (1 - cos(θ)) and angular acceleration a = -ω^2 sin(θ),
    where ω^2 = g/ℓ.  We use x for the angle θ and v for the angular velocity.
    '''

    def __init__(self, omega):
        self.omega = omega

    def potential(self, x):
        return self.omega**2 * (1 - np.cos(x))

    def acceleration(self, t, x, v):
        return -self.omega**2 * np.sin(x)

In [ ]:
class FirstOrderODEIntegrator:
    '''
    An integrator is an algorithm for producing numerical solutions to ODEs.
    '''

    def __init__(self, ODE):

        self.ODE = ODE

    def starting_from(self, t_0, y_0):
        raise NotImplementedError('No integration algorithm has been specified.')

In [ ]:
class VelocityVerlet(FirstOrderODEIntegrator):
    '''
    Velocity Verlet splits each step into a half kick, a drift, and another half kick,

    v_{n+1/2} = v_n       + 1/2 a(t_n, x_n) Δt
    x_{n+1}   = x_n       + v_{n+1/2} Δt
    v_{n+1}   = v_{n+1/2} + 1/2 a(t_{n+1}, x_{n+1}) Δt

    It needs the ODE's acceleration, so it only works on second-order ODEs.
    '''

    def __init__(self, ODE, Delta_t, steps):

        self.Delta_t = Delta_t
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, t_0, y_0):

        # Populate all the t values
        t = t_0 + self.Delta_t * np.arange(self.steps+1)
        # and allocate an array with room for a whole state at each time.
        y = np.zeros((self.steps+1,) + np.shape(y_0))

        # The user picked a starting state
        y[0] = y_0

        for step in range(self.steps):
            x, v = y[step]
            # (acceleration also wants a velocity; we hand it the newest one we have.
            #  For forces that depend only on position it doesn't matter.)
            v_half = v      + 0.5 * self.ODE.acceleration(t[step],   x,     v)      * self.Delta_t   # half kick
            x_new  = x      +       v_half                                           * self.Delta_t   # drift
            v_new  = v_half + 0.5 * self.ODE.acceleration(t[step+1], x_new, v_half) * self.Delta_t   # half kick
            y[step+1] = np.array([x_new, v_new])

        return t, y

# Pendulums

Just like in class, we start a handful of pendulums at the bottom ($\theta = 0$), each with a different energy.  At the bottom all of the energy is kinetic, so each one's starting angular velocity is $\sqrt{2E}$.

In [ ]:
pendulum = Pendulum(1.0)

E = np.array([0.25, 0.5, 1.0, 1.5, 1.9, 2.5])
y_0 = np.array([np.zeros_like(E), np.sqrt(2 * E)])

Delta_t = 0.001
steps = 20000
t, y = VelocityVerlet(pendulum, Delta_t, steps).starting_from(0, y_0)

print(t.shape)
print(y.shape)

The axes of `y` are (time, state, pendulum).
 - `y[n]` is everything at time `t[n]`.
 - Along the state axis, 0 is the angle $\theta$ and 1 is the angular velocity.
 - Along the pendulum axis, pendulum `k` is the one with energy `E[k]`.

Remember: predict each shape *before* you run the cell.  The state axis always has length 2, so it's fine to just write `2` for it.

In [ ]:
starting_states = ...                # the whole state of every pendulum at the start
check_shape(starting_states, ...)

every_angle = ...                    # every pendulum's angle at every time
check_shape(every_angle, ...)

pendulum_3_velocity = ...            # the angular velocity of pendulum 3 (with energy E[3]) at every time
check_shape(pendulum_3_velocity, ...)

last_pendulum_at_the_end = ...       # the whole state of the last pendulum at the last time
check_shape(last_pendulum_at_the_end, ...)

Times aren't indices.  To find pendulum 0's angle at $t = 5$ we first need to figure out *which step* happened at $t=5$.

In [ ]:
step = ... # the step that lands on t = 5.  Use Delta_t!
print(t[step])   # should be 5, give or take a tiny rounding error

angle_at_5 = ... # pendulum 0's angle at t = 5
check_shape(angle_at_5, ...)
print(angle_at_5)

Every pendulum started with exactly its energy `E`, so the energies at the start should match `E`.  Compute every pendulum's energy at every time, and check.

In [ ]:
energy = ... # every pendulum's energy at every time, using pendulum.energy(x, v)
check_shape(energy, ...)

print(E)
print(energy[0])

Velocity Verlet doesn't conserve energy *exactly*, but it doesn't drift either.  How far off is each pendulum's energy at the end?

In [ ]:
change = ... # each pendulum's final energy minus its starting energy
check_shape(change, ...)
print(change)

# One Array, Many Matrices

Here's where "which axes are the matrix?" really falls apart.  We're going to make three different plots out of the same `y`, by asking three different questions.

First: every pendulum's angle as a function of time.  `plt.plot(t, ...)` needs a 2D array whose first axis is time, and it will draw one curve for each entry along the second axis.

In [ ]:
fig, ax = plt.subplots()

... # one curve per pendulum: its angle vs. time

ax.set_xlabel('t')
ax.set_ylabel('θ')

Second: the phase-space trajectory of pendulum 2 alone, angular velocity vs. angle.

In [ ]:
fig, ax = plt.subplots()

... # pendulum 2's angular velocity vs. its angle

ax.set_aspect(1)
ax.set_xlabel('θ')
ax.set_ylabel('angular velocity')

Third: every pendulum's phase-space trajectory, all at once, in one call to `ax.plot`.  (Hint: if *both* arguments to `plt.plot` are 2D, it pairs up their columns.)

In [ ]:
fig, ax = plt.subplots()

... # every pendulum's angular velocity vs. its angle, one curve each

ax.set_xlabel('θ')
ax.set_ylabel('angular velocity')

**Without running it: what would `ax.plot(t, y[:, :, 2])` draw?  How many curves, and what are they?**

**And what goes wrong with `ax.plot(t, y[0])`?**

# Ranges

Ranges keep their axis but make it shorter.  Remember that `a:b` stops *before* `b`, and a missing `a` means "from the start" while a missing `b` means "to the end."

In [ ]:
every_tenth = ...                    # every 10th time step, everything else
check_shape(every_tenth, ...)        # careful!  How many steps are there, and does 0 count?

first_half = ...                     # everything from the first half of the run
check_shape(first_half, ...)

last_three = ...                     # everything about the last three pendulums
check_shape(last_three, ...)

Here's a subtle one.  These two hold exactly the same numbers.  Predict both shapes.

In [ ]:
check_shape(y[:, :, 2],   ...)
check_shape(y[:, :, 2:3], ...)

An index *removes* its axis, but a range *keeps* it, even if the range only has one thing in it.

Suppose we want to compare every pendulum with pendulum 2, `y - (pendulum 2)`.  Only one of these works.  Predict which before you run it!

In [ ]:
try:
    difference = y - y[:, :, 2:3]
    print('2:3 works', difference.shape)
except ValueError as error:
    print('2:3 fails:', error)

try:
    difference = y - y[:, :, 2]
    print('2 works', difference.shape)
except ValueError as error:
    print('2 fails:', error)

**Explain, using the broadcasting rules, why one works and the other doesn't.**

# Writing Into Slices

Anything you can read with a slice you can also *write*, by putting the slice on the left of `=`.  The right-hand side has to fit the slice's shape, by the broadcasting rules.  A single number fits anything.

Let's rebuild the pendulums' starting states by writing into an array of zeros.  The starting angles are already right (they're 0), so we only need the angular velocities, $\sqrt{2E}$.  Do it in one line, with no loop.

In [ ]:
start = np.zeros((2, len(E)))   # (state, pendulum)

... # write every pendulum's starting angular velocity into start

print(np.array_equal(start, y_0))   # should print True

What if the right-hand side doesn't fit?

In [ ]:
try:
    start[1] = np.array([1.0, 2.0, 3.0])
except ValueError as error:
    print(error)

**In your own words, why does that fail?**

A different experiment.  This time every pendulum starts *at rest*, pulled back to an angle of 1 radian, except that the last pendulum also gets a shove: it starts with angular velocity 2.

Fill in `released` with two writes, and *no* arrays on the right-hand side, just numbers.

In [ ]:
released = np.zeros((2, len(E)))   # (state, pendulum)

... # every pendulum starts at angle 1
... # the last pendulum starts with angular velocity 2

print(released)

t, y_released = VelocityVerlet(pendulum, Delta_t, steps).starting_from(0, released)

fig, ax = plt.subplots()
ax.plot(t, y_released[:, 0])
ax.set_xlabel('t')
ax.set_ylabel('θ')

All the pendulums but one should be swinging in lockstep, right on top of each other.  (You only see one color for those, because each one is drawn right on top of the one before.)

Now look back at `VelocityVerlet.starting_from` in the Setup.  It writes into slices too:
```python
y = np.zeros((self.steps+1,) + np.shape(y_0))
y[0] = y_0
...
y[step+1] = np.array([x_new, v_new])
```

**What's the shape of `y[step+1]` in terms of `y_0`?  Why does the integrator work no matter how many pendulums we start?**

One more write.  Our fastest pendulum goes over the top, so its angle keeps growing forever.  For a phase-space picture we'd rather keep every angle between $-\pi$ and $\pi$.  Make a copy of `y` (so we don't change the original), and overwrite *only the angles* in the copy with
\begin{align}
    \theta \to \left( (\theta + \pi) \bmod 2\pi \right) - \pi
\end{align}
using [`np.mod`](https://numpy.org/doc/stable/reference/generated/numpy.mod.html).

In [ ]:
wrapped = y.copy()

... # wrap every angle, at every time, for every pendulum, into the range from -π to π

fig, ax = plt.subplots()
ax.plot(wrapped[:, 0], wrapped[:, 1], linestyle='none', marker=',')
ax.set_xlabel('θ')
ax.set_ylabel('angular velocity')

# A Grid of Pendulums

So far our pendulums have been lined up along one axis.  But nothing says they have to be!

Let's start a pendulum at *every combination* of a few starting angles and a few starting angular velocities.  Then the starting state needs three axes: (state, starting angle, starting velocity).

In [ ]:
theta_0 = np.linspace(-3, 3, 7)
v_0 = np.linspace(-2, 2, 5)

grid_start = np.zeros((2, len(theta_0), len(v_0)))   # (state, starting angle, starting velocity)

Fill in `grid_start` with two writes, so that pendulum `[i, j]` starts at angle `theta_0[i]` with angular velocity `v_0[j]`.  You'll need `np.newaxis` for one of them!

In [ ]:
... # every starting angle: grid_start[0, i, j] should be theta_0[i]
... # every starting velocity: grid_start[1, i, j] should be v_0[j]

# Check every pendulum, the slow way.
print(all(
    grid_start[0, i, j] == theta_0[i] and grid_start[1, i, j] == v_0[j]
    for i in range(len(theta_0))
    for j in range(len(v_0))
))

And now: the same integrator, completely unchanged.

In [ ]:
t, grid = VelocityVerlet(pendulum, Delta_t, steps).starting_from(0, grid_start)
print(grid.shape)

`grid` has four axes, (time, state, starting angle, starting velocity).  Four questions to answer to get one number.

# `...`

`...` means "all of them" for *every* axis you don't mention.  These three are all the same thing:

In [ ]:
a = grid[:, 0, :, :]
b = grid[:, 0, ...]
c = grid[:, 0]

print(a.shape, b.shape, c.shape)
print(np.array_equal(a, b) and np.array_equal(b, c))

When the axes you care about are on the left, `...` is optional.  It's when they're on the *right* that it earns its keep, because `...` can go first.

In [ ]:
first_velocity = ...                  # everything about the pendulums that started with angular velocity v_0[0], using ...
check_shape(first_velocity, ...)

final_states = ...                   # the whole state of every pendulum at the last time
check_shape(final_states, ...)

one_pendulum = ...                   # the angle of the pendulum that started at theta_0[1] with v_0[3], at every time
check_shape(one_pendulum, ...)

Here's where `...` really shines: code that doesn't know (or care) how many axes it's going to get.

Write a function that computes a pendulum's energy from its state.  The state's **first** axis is (angle, angular velocity), and it can have any number of other axes after that, meaning whatever they mean.

You don't need to write out the energy formula again: `pendulum.energy(x, v)` already does that.  Your job is just to pull the angle and angular velocity out of `state` and hand them over.

In [ ]:
def energy_of(state):
    '''
    The pendulum's energy.  The first axis of state is (angle, angular velocity);
    there can be any number of other axes after it.
    '''
    raise NotImplementedError("This function should split state into its angle and angular velocity, and pass them to pendulum.energy.")

print(energy_of(np.array([0.0, 1.0])))        # one pendulum, should be 0.5
print(energy_of(y_0) - E)                     # a row of pendulums, should be zeros
print(energy_of(grid_start).shape)            # a grid of pendulums
print(np.max(np.abs(energy_of(grid[-1]) - energy_of(grid[0]))))   # small!

This is exactly what `x, v = y` does in `SecondOrderODE.f`: unpacking splits an array along its *first* axis, so `x` is `y[0, ...]` and `v` is `y[1, ...]`.  That's why `f` (and so the integrator) works on any batch of pendulums.

But careful.  What does `energy_of(grid)` compute?

In [ ]:
print(energy_of(grid).shape)

**What is `energy_of(grid)` actually computing, and why doesn't numpy complain?  What should we write instead to get every pendulum's energy at every time?**

Sometimes the axis we want is the **last** one, and then `...` is required.  For example, 3D vectors are usually stored with $(x, y, z)$ on the last axis: a single velocity has shape `(3,)`, a hundred of them have shape `(100, 3)`, and so on.

Write a function that computes the speed $\sqrt{v_x^2 + v_y^2 + v_z^2}$ for any number of velocities, arranged however you like.  Index the components; don't use `np.sum` or `np.linalg.norm`.

In [ ]:
def speed(velocity):
    '''
    The length of 3D velocity vectors.  The last axis of velocity is (x, y, z);
    there can be any number of other axes before it.
    '''
    raise NotImplementedError("This function should compute the speed of every velocity in the array.")

print(speed(np.array([3.0, 4.0, 12.0])))           # should be 13
print(speed(rng.normal(size=(100, 3))).shape)
print(speed(rng.normal(size=(4, 100, 3))).shape)

**Why wouldn't `velocity[:, 0]` work instead of `velocity[..., 0]`?  Try it on each of the three examples above.**

# Axes That Are Space

All of our axes so far have been bookkeeping: which time, which pendulum, which station.  But sometimes the axes really are $x$, $y$, and $z$!

If we evaluate a function at every point of a 3D grid, we need three indices, because a point needs three coordinates.  `np.meshgrid` makes the grid for us.

In [ ]:
x = np.linspace(-2, 2, 41)
y = np.linspace(-1.5, 1.5, 31)   # we're done with the pendulums, so we can reuse the name y
z = np.linspace(-2, 2, 21)

X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
print(X.shape)

With `indexing='ij'`, `X[i, j, k]` is `x[i]`, `Y[i, j, k]` is `y[j]`, and `Z[i, j, k]` is `z[k]`.  So the axes are ($x$, $y$, $z$), in that order.

(Watch out: *without* `indexing='ij'`, `np.meshgrid` swaps the first two axes, to match the rows-are-$y$ convention of plotting.  That's a classic source of confusion.)

In [ ]:
print(np.meshgrid(x, y, z, indexing='ij')[0].shape)
print(np.meshgrid(x, y, z)[0].shape)

Let's put a positive charge at $(0, 0, +\tfrac{1}{2})$ and a negative charge at $(0, 0, -\tfrac{1}{2})$, a dipole.  In units where $kq = 1$ the electric potential is
\begin{align}
    \phi &= \frac{1}{r_+} - \frac{1}{r_-}
\end{align}
where $r_\pm$ is the distance to each charge.

In [ ]:
r_plus  = np.sqrt(X**2 + Y**2 + (Z - 0.5)**2)
r_minus = np.sqrt(X**2 + Y**2 + (Z + 0.5)**2)

phi = 1 / r_plus - 1 / r_minus
print(phi.shape)

To find a point we need its indices.  This function finds the index of the grid point closest to a value.

In [ ]:
def index_of(grid, value):
    '''
    The index of the entry of grid closest to value.
    '''
    return np.argmin(np.abs(grid - value))

print(index_of(x, 1.0), x[index_of(x, 1.0)])

On the $z$ axis above both charges, at $(0, 0, 1)$, the potential should be $1/\tfrac{1}{2} - 1/\tfrac{3}{2} = \tfrac{4}{3}$.

In [ ]:
on_axis = ... # phi at the point (0, 0, 1)
check_shape(on_axis, ...)
print(on_axis)

Now a line: the potential all along the $z$ axis, where $x = y = 0$.

In [ ]:
along_z = ... # phi at every point on the z axis
check_shape(along_z, ...)

fig, ax = plt.subplots()
ax.plot(z, along_z, marker='.')
ax.set_xlabel('z')
ax.set_ylabel('φ')

Now a plane: the potential everywhere in the $xz$ plane, where $y = 0$.  To draw a contour plot we also need to know *where* each value lives, so slice `X` and `Z` the same way as `phi`.

In [ ]:
phi_xz = ... # phi everywhere in the plane y = 0
X_xz   = ... # the x coordinate of each of those points
Z_xz   = ... # the z coordinate of each of those points
check_shape(phi_xz, ...)

fig, ax = plt.subplots(figsize=(6, 6))
ax.contourf(X_xz, Z_xz, phi_xz, levels=np.linspace(-3, 3, 25), cmap='RdBu_r', extend='both')
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('z')

Finally the $xy$ plane, where $z = 0$, halfway between the charges.  Grab it, and print the largest absolute value of the potential anywhere in that plane.

In [ ]:
phi_xy = ... # phi everywhere in the plane z = 0
check_shape(phi_xy, ...)

print(np.max(np.abs(phi_xy)))

**Explain the number you got, physically.  What do the two axes of `phi_xy` mean?**